# 05. Granice (GUGiK ULDK)

Województwa po kodzie TERYT, od razu we współrzędnych geograficznych. Bez konta.
`pip install shapely matplotlib`

In [ ]:
import requests
import matplotlib.pyplot as plt
import shapely
from shapely import wkt

ULDK = "https://uldk.gugik.gov.pl/"

In [ ]:
# granice: 16 wojewodztw (kody TERYT 02, 04, ..., 32)
voivodeships = {}
for code in [f"{n:02d}" for n in range(2, 33, 2)]:
    params = {"request": "GetVoivodeshipById", "id": code, "result": "teryt,voivodeship,geom_wkt", "srid": 4326}
    lines = requests.get(ULDK, params=params).text.strip().split("\n")
    teryt, name, geom = lines[1].split("|")
    voivodeships[teryt] = (name, wkt.loads(geom.split(";", 1)[1]))

for teryt, (name, geom) in voivodeships.items():
    print(teryt, name, "| punktów granicy:", shapely.get_num_coordinates(geom))

Odpowiedź to tekst: pierwsza linia `0` = OK, druga `teryt|nazwa|SRID=4326;MULTIPOLYGON(...)`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for teryt, (name, geom) in voivodeships.items():
    for polygon in getattr(geom, "geoms", [geom]):
        ax.plot(*polygon.exterior.xy, color="black", lw=0.7)
    ax.text(geom.centroid.x, geom.centroid.y, teryt, ha="center")
plt.show()

Przypisanie punktu do powiatu jednym zapytaniem (np. stacja IMGW albo punkt pogody):

In [ ]:
params = {"request": "GetCountyByXY", "xy": "20.96,52.16,4326", "result": "teryt,county,voivodeship"}
requests.get(ULDK, params=params).text